# The flavor puzzle: thirteen numbers the SM fits but does not explain

The Standard Model reproduces every quark and charged-lepton mass and every CKM mixing, but only
because each one is a free parameter. Nothing in the theory says why the top quark is heavy and
the electron light, or why quark mixing is small. This notebook makes that statement
**quantitative**. It turns "the SM does not explain the hierarchy" into one number computed from
the model itself, the number a flavor model will later have to lower (P2/P3).

Each section runs four moves: **set up** (look at the raw object), **collect** (let the structure
emerge), **recognise** (name what appeared), and **check** (by an independent route). Before the
cell that settles each question, a blockquote asks you to commit to an expectation.

**The one question this notebook is really about.** How big is the hierarchy the SM leaves
unexplained, measured in a way that a theory of flavor could improve on?

*Housekeeping.* Data, sources and status live in [`puzzle.yaml`](puzzle.yaml). They are loaded
below and never re-typed as literals. Decisions are logged in [`decisions.md`](decisions.md).
The notebook ships already solved; to use it as an exercise, make a copy before reading
`solutions/`.

In [1]:
import math
import sys
from pathlib import Path

import numpy as np
import sympy as sp
from IPython.display import Markdown, Math, display

# nbclient/nbmake set the kernel's cwd to this notebook's own directory; make sure it (and the
# repo root two levels up, for the `puzzles`/`checkpoints`/`fit`/`feynlag_anomalies` packages)
# are importable regardless of how the kernel was launched.
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parents[1]
for p in (NB_DIR, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from feynlag_anomalies.latex import latex
from feynlag_anomalies.registry import load_puzzle
from puzzles.flavor_puzzle import checkpoints_def as cd

sp.init_printing(use_latex="mathjax", latex_printer=latex)


def ok(msg):
    print("✓", msg)


def trace(msg, obj=None):
    print(msg)
    if obj is not None:
        display(obj)


puzzle = load_puzzle("flavor_puzzle")
q = {s.name: s for s in puzzle.sm_quantities}
print(puzzle.title, "--", puzzle.maturity)

The flavor puzzle: hierarchical masses and mixings -- P1


---
## 1. The SM has thirteen flavor parameters and fixes none of them

With three generations, the charged-fermion sector has nine masses, three CKM angles and one CP
phase *(cited: PDG 2024 CKM review, §12.1)*. The SM's gauge interactions do not distinguish the
generations; only the Yukawa couplings do, and their values are inputs.

Masses run with the renormalization scale, so they must be compared at **one** scale. We use
MS-bar masses at $\mu = M_Z$ in the full SM (Huang & Zhou), and the PDG global-fit CKM angles,
which barely run below the weak scale *(cited: PDG 2024, §12.1)*.

> **Before running the next cell.** The top quark weighs about 170 GeV and the electron about
> half an MeV. How many orders of magnitude is that? Guess before you look.

In [2]:
# ---- MOVE 1: set up.  The sourced parameters, straight from puzzle.yaml. ----
rows = ["| quantity | value | 1σ | units | source |", "|---|---|---|---|---|"]
for s in puzzle.sm_quantities:
    unc = "—" if s.uncertainty is None else f"{s.uncertainty:.2g}"
    rows.append(f"| {s.name} | {s.value:.5g} | {unc} | {s.units} | {s.source} |")
consulted = ", ".join(f"{src.identifier} ({src.consulted_on})" for src in puzzle.sources)
display(Markdown("\n".join(rows) + f"\n\nSources consulted: {consulted}."))

ratio = q["m_t(M_Z)"].value / q["m_e(M_Z)"].value
print(f"m_t / m_e at M_Z = {ratio:.3g}  ->  {math.log10(ratio):.2f} orders of magnitude")
assert ratio > 1e5
ok("the charged-fermion masses span more than five orders of magnitude")

| quantity | value | 1σ | units | source |
|---|---|---|---|---|
| m_u(M_Z) | 0.00123 | 0.00021 | GeV | arXiv:2009.04851 |
| m_c(M_Z) | 0.62 | 0.017 | GeV | arXiv:2009.04851 |
| m_t(M_Z) | 168.26 | 0.75 | GeV | arXiv:2009.04851 |
| m_d(M_Z) | 0.00267 | 0.00019 | GeV | arXiv:2009.04851 |
| m_s(M_Z) | 0.05316 | 0.0046 | GeV | arXiv:2009.04851 |
| m_b(M_Z) | 2.839 | 0.026 | GeV | arXiv:2009.04851 |
| m_e(M_Z) | 0.00048307 | 4.5e-07 | GeV | arXiv:2009.04851 |
| m_mu(M_Z) | 0.10177 | 2.3e-05 | GeV | arXiv:2009.04851 |
| m_tau(M_Z) | 1.7286 | 0.00028 | GeV | arXiv:2009.04851 |
| v_F | 246 | — | GeV | arXiv:2009.04851 |
| sin theta_12 | 0.22501 | 0.00068 | 1 | PDG2024:rev-ckm-matrix |
| sin theta_13 | 0.003732 | 8.7e-05 | 1 | PDG2024:rev-ckm-matrix |
| sin theta_23 | 0.04183 | 0.00074 | 1 | PDG2024:rev-ckm-matrix |
| delta | 1.147 | 0.026 | rad | PDG2024:rev-ckm-matrix |

Sources consulted: arXiv:2009.04851 (2026-10-04), PDG2024:rev-ckm-matrix (2026-10-04).

m_t / m_e at M_Z = 3.48e+05  ->  5.54 orders of magnitude
✓ the charged-fermion masses span more than five orders of magnitude


**Read it.** Five and a half orders of magnitude between the heaviest and lightest charged
fermion *(computed from the sourced masses)*. The CKM sines, by contrast, all sit between
$10^{-3}$ and $1$. But a mass ratio is not yet a statement about the *theory*: the masses come
from dimensionless Yukawa couplings times the Higgs vev, and it is the couplings the SM leaves
free. The next section reads them from the model.

---
## 2. In the SM a Yukawa is a mass divided by the vev

The flavor model here is feynlag-models' `sm_ckm` (maturity L3): three generations, with the CKM
matrix in the charged current. It is built at a benchmark whose masses, vev and CKM angles are
the sourced values above, so every number below comes from the model, not from a formula typed
in this notebook.

> **Before running the next cell.** The top mass is about $v/\sqrt2$. What should $y_t$ be, and
> what does that say about whether the top Yukawa is "unnatural"?

In [3]:
# ---- MOVE 1: set up.  Build sm_ckm at the sourced benchmark. ----
from feynlag_models.registry import build, metadata
from puzzles.flavor_puzzle.solutions.step_1 import MODEL_ID, sm_benchmark

print(f"{MODEL_ID}: feynlag-models maturity L{metadata(MODEL_ID)['maturity_level']}")
bundle = build(MODEL_ID, benchmark=sm_benchmark(puzzle))
ys = bundle.pieces.yukawa_params
display(Math(latex(ys["yt"].symbol) + " = " + latex(ys["yt"].expr)))

sm_ckm: feynlag-models maturity L3


<IPython.core.display.Math object>

In [4]:
# ---- MOVE 2: collect.  Evaluate the model's own relation at its benchmark. ----
values = bundle.params.numeric()
answer_1 = values[ys["yt"].symbol]
check_1 = cd.make_check_1(bundle)
assert check_1(answer_1)

[step_1] correct -- got 0.967299


In [5]:
# ---- peek: all nine Yukawas, sector by sector. ----
from puzzles.flavor_puzzle.solutions.step_1 import SECTORS, sm_yukawas

yukawas = sm_yukawas(bundle)
rows = ["| sector | gen 1 | gen 2 | gen 3 |", "|---|---|---|---|"]
for sector, names in SECTORS.items():
    rows.append(f"| {sector} | " + " | ".join(
        f"${latex(ys[n].symbol)}$ = {yukawas[n]:.2e}" for n in names) + " |")
display(Markdown("\n".join(rows)))
assert yukawas["yt"] > 0.5 and yukawas["ye"] < 1e-5
ok("y_t is O(1); every other Yukawa is small, y_e by almost six orders of magnitude")

| sector | gen 1 | gen 2 | gen 3 |
|---|---|---|---|
| up | ${y_u}$ = 7.07e-06 | ${y_c}$ = 3.56e-03 | ${y_t}$ = 9.67e-01 |
| down | ${y_d}$ = 1.53e-05 | ${y_s}$ = 3.06e-04 | ${y_b}$ = 1.63e-02 |
| lepton | ${y_e}$ = 2.78e-06 | ${y_\mu}$ = 5.85e-04 | ${y_\tau}$ = 9.94e-03 |

✓ y_t is O(1); every other Yukawa is small, y_e by almost six orders of magnitude


**Recognise.** $y_t \approx 0.97$ *(computed)*. The top Yukawa is the one coupling that looks
"natural". The puzzle is everyone else: eight couplings between $2\times10^{-2}$ and $3\times10^{-6}$,
with nothing in the SM to make them small. Small dimensionless couplings are technically natural
(chiral symmetry protects them), so this is not a fine-tuning problem; it is an *explanation*
problem: the pattern is not random-looking, yet the SM has no structure that produces it.

---
## 3. The SM's free flavor parameters span five and a half orders of magnitude

To compare the SM with a flavor model we need one number both can be scored on. The quantifier
`free_parameter_log10_span` is $\log_{10}(\max|c| / \min|c|)$ over the dimensionless free
parameters $c$ a theory needs to reproduce the masses and the CKM matrix. In the SM those are
the nine Yukawas and the three CKM sines; the phase $\delta$ is an angle with no hierarchy content
and is left out (see `decisions.md`).

> **Before running the next cell.** Which two parameters set the span? And which sector, up,
> down or lepton, has the widest spread on its own?

In [6]:
# ---- MOVE 2: collect.  The quantifier over the SM's free parameters. ----
from puzzles.flavor_puzzle.solutions.step_1 import (
    free_parameter_log10_span, sector_spans, sm_free_parameters,
)

params = sm_free_parameters(bundle)
answer_2 = free_parameter_log10_span(params)
check_2 = cd.make_check_2(bundle)
assert check_2(answer_2)

[step_2] correct -- got 5.54197


In [7]:
# ---- MOVE 3 + 4: recognise the endpoints, check against puzzle.yaml. ----
spans = sector_spans(yukawas)
print(f"  largest  : {max(params, key=params.get)}   smallest : {min(params, key=params.get)}")
for sector, span in spans.items():
    print(f"  {sector:<7} span = {span:.2f} orders of magnitude")
assert abs(answer_2 - puzzle.quantifier.sm_value) < 5e-4
ok(f"the SM quantifier, {answer_2:.3f} orders of magnitude, matches puzzle.yaml")

  largest  : yt   smallest : ye
  up      span = 5.14 orders of magnitude
  down    span = 3.03 orders of magnitude
  lepton  span = 3.55 orders of magnitude
✓ the SM quantifier, 5.542 orders of magnitude, matches puzzle.yaml


**The judgment to keep:** the SM scores about **5.5 orders of magnitude** *(computed)*, set
entirely by $y_t/y_e$. The up sector alone spans over five orders of magnitude, more than the down
quarks or the leptons. A theory of flavor "addresses" the puzzle to the extent that its *own* free parameters
span less while still fitting the data. That is the bar for P3, and it has to be met with a
computed number, not an argument.

---
## 4. Quark mixing is hierarchical in powers of the Cabibbo angle

The CKM matrix is close to the identity. Wolfenstein's parametrization makes the pattern explicit:
with $\lambda = s_{12}$, the other angles go as $s_{23} = A\lambda^2$ and $s_{13} \sim A\lambda^3$,
with $A$ and $\bar\rho, \bar\eta$ of order one *(cited: PDG 2024, Eq. 12.4)*.

> **Before running the next cell.** If $s_{23} = A\lambda^2$, what is $A$? Is it $O(1)$, or is
> the hierarchy hiding there too?

In [8]:
# ---- MOVE 1 + 2: the CKM matrix from the bundle, and its Wolfenstein parameters. ----
from puzzles.flavor_puzzle.solutions.step_2 import ckm_numeric, lambda_powers, wolfenstein

V = ckm_numeric(bundle)
rows = ["| $\\lvert V_{ij}\\rvert$ | $d$ | $s$ | $b$ |", "|---|---|---|---|"]
rows += [f"| ${u}$ | " + " | ".join(f"{abs(x):.5f}" for x in row) + " |" for u, row in zip("uct", V)]
display(Markdown("\n".join(rows)))
w = wolfenstein(V)
answer_3 = w["A"]
check_3 = cd.make_check_3(bundle)
assert check_3(answer_3)
print("  " + ", ".join(f"{k} = {v:.4f}" for k, v in w.items()))

| $\lvert V_{ij}\rvert$ | $d$ | $s$ | $b$ |
|---|---|---|---|
| $u$ | 0.97435 | 0.22501 | 0.00373 |
| $c$ | 0.22488 | 0.97349 | 0.04183 |
| $t$ | 0.00858 | 0.04111 | 0.99912 |

[step_3] correct -- got 0.826198
  lambda = 0.2250, A = 0.8262, rho_bar = 0.1592, eta_bar = 0.3523


In [9]:
# ---- MOVE 3 + 4: recognise the powers of lambda; check unitarity independently. ----
lam = w["lambda"]
n_ckm = lambda_powers({k: params[k] for k in ("s12", "s23", "s13")}, lam)
print("  CKM sines as lambda^n :", ", ".join(f"{k}: n = {n:.2f}" for k, n in n_ckm.items()))
assert np.allclose(V.conj().T @ V, np.eye(3), atol=1e-12)
assert 0.5 < w["A"] < 2 and 0.1 < abs(complex(w["rho_bar"], w["eta_bar"])) < 1
ok("V is unitary, and once powers of lambda are pulled out, A, rho_bar, eta_bar are O(1)")

  CKM sines as lambda^n : s12: n = 1.00, s23: n = 2.13, s13: n = 3.75
✓ V is unitary, and once powers of lambda are pulled out, A, rho_bar, eta_bar are O(1)


**Recognise.** $A \approx 0.83$ *(computed)*; the regression in `tests/test_step2_ckm.py`
confirms that $\lambda, A, \bar\rho, \bar\eta$ from this matrix fall inside the PDG 2024 fit bands
(Eq. 12.26). So the CKM hierarchy *is* a power law in one small number, with $O(1)$ coefficients.

Does the same work for the masses? Write each Yukawa as $\lambda^{n_f}$ and look at the exponents.

In [10]:
# ---- peek: the Yukawas as powers of lambda -- the bridge to Froggatt-Nielsen. ----
n_y = lambda_powers(yukawas, lam)
rows = ["| sector | gen 1 | gen 2 | gen 3 |", "|---|---|---|---|"]
for sector, names in SECTORS.items():
    rows.append(f"| {sector} | " + " | ".join(
        f"${latex(ys[n].symbol)} \\sim \\lambda^{{{n_y[n]:.1f}}}$" for n in names) + " |")
display(Markdown("\n".join(rows)))

| sector | gen 1 | gen 2 | gen 3 |
|---|---|---|---|
| up | ${y_u} \sim \lambda^{8.0}$ | ${y_c} \sim \lambda^{3.8}$ | ${y_t} \sim \lambda^{0.0}$ |
| down | ${y_d} \sim \lambda^{7.4}$ | ${y_s} \sim \lambda^{5.4}$ | ${y_b} \sim \lambda^{2.8}$ |
| lepton | ${y_e} \sim \lambda^{8.6}$ | ${y_\mu} \sim \lambda^{5.0}$ | ${y_\tau} \sim \lambda^{3.1}$ |

**The judgment to keep:** every Yukawa is $\lambda^{n}$ with $0 \lesssim n \lesssim 9$
*(computed)*. The exponents are not integers, but they do not have to be: an integer power times
an $O(1)$ coefficient covers the gap. That is exactly the Froggatt–Nielsen idea. A $U(1)$ flavor
charge for each field makes $y_f \sim c_f\,\epsilon^{n_f}$ with $\epsilon \approx \lambda$ and
$c_f = O(1)$, so the free parameters become the $c_f$, whose span should be far below 5.5
orders of magnitude. Testing that is P2 (model request) and P3 (fit), not this notebook.

---
## 5. Recap

**The numbers**, each one produced by a cell above:

In [11]:
summary = [
    ("1", "$m_t/m_e$ at $M_Z$", f"{ratio:.2e}", "computed (masses cited: arXiv:2009.04851)"),
    ("2", "$y_t$ from `sm_ckm`", f"{answer_1:.4f}", "computed"),
    ("3", "SM quantifier (free-parameter span)", f"{answer_2:.3f} orders of magnitude", "computed"),
    ("3", "sector spans up / down / lepton",
     " / ".join(f"{spans[s]:.2f}" for s in ("up", "down", "lepton")), "computed"),
    ("4", "Wolfenstein $\\lambda$, $A$", f"{w['lambda']:.5f}, {w['A']:.3f}",
     "computed (inputs cited: PDG 2024 Eq. 12.28)"),
]
rows = ["| § | result | key number | provenance |", "|---|---|---|---|"]
rows += [f"| {s} | {r} | {n} | {p} |" for s, r, n, p in summary]
display(Markdown("\n".join(rows)))

| § | result | key number | provenance |
|---|---|---|---|
| 1 | $m_t/m_e$ at $M_Z$ | 3.48e+05 | computed (masses cited: arXiv:2009.04851) |
| 2 | $y_t$ from `sm_ckm` | 0.9673 | computed |
| 3 | SM quantifier (free-parameter span) | 5.542 orders of magnitude | computed |
| 3 | sector spans up / down / lepton | 5.14 / 3.03 / 3.55 | computed |
| 4 | Wolfenstein $\lambda$, $A$ | 0.22501, 0.826 | computed (inputs cited: PDG 2024 Eq. 12.28) |

**The mechanics.**

1. Thirteen flavor parameters, all inputs; compare masses at one scale ($\mu = M_Z$).
2. In `sm_ckm`, $y_f = \sqrt2\, m_f/v$: the hierarchy lives in dimensionless couplings.
3. The quantifier, $\log_{10}(\max/\min)$ of the free parameters, is 5.5 orders of magnitude
   in the SM.
4. The CKM hierarchy is a power law in $\lambda$ with $O(1)$ coefficients; the Yukawas are
   $\lambda^{n}$ with non-integer $n$, which is what Froggatt–Nielsen addresses.

**The tools, in the order you should reach for them.**

| question | tool |
|---|---|
| what does the model say a Yukawa is? | `bundle.pieces.yukawa_params[name].expr` |
| its value at the sourced benchmark? | `build("sm_ckm", benchmark=sm_benchmark(puzzle))`, `bundle.params.numeric()` |
| the CKM matrix? | `bundle.extra["V_expr"]` evaluated at the benchmark |
| how hierarchical? | `free_parameter_log10_span`, `lambda_powers` |

**Traps.**

- *Mixing scales.* A pole mass for the top and an MS-bar mass at 2 GeV for the strange quark do
  not belong in one ratio. Use one scheme and one scale.
- *The vev convention.* Huang & Zhou define $v_F = 246$ GeV exactly; `sm_ckm`'s default is
  246.22 GeV. The Yukawas here use the source's convention, so they match its tables.
- *Reading too much into the quantifier.* It counts orders of magnitude, not parameters, and it
  says nothing about whether $O(1)$ coefficients are themselves tuned. See `decisions.md`.
- *Natural vs. explained.* Small Yukawas are technically natural; the flavor puzzle is about the
  missing explanation, not about fine-tuning.

**Next steps.** P2: file `model_requests/froggatt_nielsen.md` (needs approval). P3: compute the
same quantifier for the model's $O(1)$ coefficients and confront it with the data.